# Whisper-small AdaLoRA Fine-tuning — HDSD (Hindi Dysarthric Speech)

**Before running:** In Colab menu → Runtime → Change runtime type → **GPU (T4)**.

**Expected upload layout in Google Drive:**
```
MyDrive/
  HDSD/
    colab_train.py
    hdsd_augment.py
    train.csv  val.csv  test.csv
    hindi_sent/
      F11/  F12/  ...   (WAV files)
```

Run cells **top to bottom**. Training (Cell 8) takes ~4–7 h on T4.  
Checkpoints save every 250 steps → safe to let the session reconnect.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → GPU (T4).")
print("GPU:", result.stdout.strip())
print("Python:", sys.version.split()[0])

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Pin transformers to ≥ 4.46 (eval_strategy + processing_class are available)
# peft ≥ 0.10 for stable AdaLoRA; accelerate for fp16 + gradient checkpointing
%pip install -q \
    "transformers>=4.46.0" \
    "peft>=0.10.0" \
    "accelerate>=0.27.0" \
    jiwer \
    soundfile \
    librosa \
    tensorboard
print("Dependencies installed.")

In [ ]:
# ── Cell 4: Copy dataset from Drive → /content/HDSD ──────────────────────────
# /content has SSD-backed storage; Drive I/O would bottleneck the dataloader.
import os, shutil, time
from pathlib import Path

DRIVE_HDSD = Path('/content/drive/MyDrive/HDSD')
LOCAL_HDSD = Path('/content/HDSD')

if not DRIVE_HDSD.exists():
    raise FileNotFoundError(
        f"{DRIVE_HDSD} not found.\n"
        "Upload your HDSD/ folder to Google Drive at MyDrive/HDSD/.")

if LOCAL_HDSD.exists():
    print(f"Skipping copy — {LOCAL_HDSD} already exists.")
else:
    print(f"Copying {DRIVE_HDSD} → {LOCAL_HDSD} (this may take a few minutes)...")
    t0 = time.time()
    shutil.copytree(str(DRIVE_HDSD), str(LOCAL_HDSD))
    elapsed = time.time() - t0
    n_wav = len(list(LOCAL_HDSD.glob('hindi_sent/**/*.wav')))
    print(f"Done in {elapsed:.0f}s — {n_wav} WAV files copied.")

# Verify required files
for f in ['colab_train.py', 'hdsd_augment.py', 'train.csv', 'val.csv', 'test.csv']:
    assert (LOCAL_HDSD / f).exists(), f"Missing: {LOCAL_HDSD / f}"
print("All required files present.")

In [ ]:
# ── Cell 5: Fix CSV audio paths → /content/HDSD ──────────────────────────────
# The CSVs may contain absolute paths from the recording machine.
# Rewrite them to point to /content/HDSD/hindi_sent/...
import re
import pandas as pd

LOCAL_HDSD = '/content/HDSD'

for csv_name in ['train.csv', 'val.csv', 'test.csv']:
    path = f'{LOCAL_HDSD}/{csv_name}'
    df = pd.read_csv(path)
    original_first = df['path'].iloc[0]
    df['path'] = df['path'].apply(
        lambda p: re.sub(r'^.+?(hindi_sent/)', f'{LOCAL_HDSD}/\\1', str(p))
    )
    df.to_csv(path, index=False)
    print(f"{csv_name}: {original_first}")
    print(f"         → {df['path'].iloc[0]}")

# Confirm a few files exist
import os
sample_paths = pd.read_csv(f'{LOCAL_HDSD}/train.csv')['path'].head(3).tolist()
for p in sample_paths:
    status = '✓' if os.path.exists(p) else '✗ MISSING'
    print(f"  {status}  {p}")

In [ ]:
# ── Cell 6: (Optional) Sanity check ──────────────────────────────────────────
# Verifies data integrity, ML stack imports, and a 3-sample forward pass.
# Skip this cell if you want to start training immediately.
%cd /content/HDSD
!python sanity_check.py --all

In [ ]:
# ── Cell 7: Configure TensorBoard (optional live monitoring) ──────────────────
%load_ext tensorboard
%tensorboard --logdir /content/HDSD/whisper-adalora-hdsd/logs

In [ ]:
# ── Cell 8: TRAIN ─────────────────────────────────────────────────────────────
# Runs ~4–7 h on T4 (2000 steps, eval every 250 steps).
# Checkpoints at whisper-adalora-hdsd/checkpoint-*/  — safe to reconnect.
# Re-running this cell after a disconnect will auto-resume from the latest ckpt.
%cd /content/HDSD
!python colab_train.py 2>&1 | tee /content/HDSD/training.log

In [ ]:
# ── Cell 9: Save results to Google Drive ──────────────────────────────────────
import shutil
from pathlib import Path

src = Path('/content/HDSD/whisper-adalora-hdsd')
dst = Path('/content/drive/MyDrive/whisper-adalora-hdsd')

if not src.exists():
    print("Output directory not found — training may not have completed.")
else:
    print(f"Copying {src} → {dst}...")
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print("Done. Saved to Google Drive.")

    # Also copy training log
    log_src = Path('/content/HDSD/training.log')
    if log_src.exists():
        shutil.copy(str(log_src), str(dst / 'training.log'))
        print("training.log saved too.")

In [ ]:
# ── Cell 10: Display results ──────────────────────────────────────────────────
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path('/content/HDSD/whisper-adalora-hdsd')

# Final test WER
wer_file = OUTPUT_DIR / 'test_wer.txt'
if wer_file.exists():
    print(wer_file.read_text())
else:
    print("test_wer.txt not found — check training.log for the final WER line.")

# Training curve (last 10 eval rows)
log_csv = OUTPUT_DIR / 'training_log.csv'
if log_csv.exists():
    df = pd.read_csv(log_csv)
    eval_rows = df[df['eval_wer'].notna()][['step', 'eval_wer', 'eval_loss']]
    print("\nEval WER curve:")
    print(eval_rows.tail(10).to_string(index=False))

    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(8, 4))
        plt.plot(eval_rows['step'], eval_rows['eval_wer'], marker='o')
        plt.axhline(15, color='red', linestyle='--', label='15% target')
        plt.xlabel('Step')
        plt.ylabel('WER (%)')
        plt.title('Validation WER — Whisper-small AdaLoRA on HDSD')
        plt.legend()
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / 'wer_curve.png'), dpi=120)
        plt.show()
        print("WER curve saved to wer_curve.png")
    except Exception as e:
        print(f"Plot skipped: {e}")

In [ ]:
# ── Cell 11: Load adapter and run inference on a single sample ─────────────────
# Useful for a quick qualitative check after training.
import torch, soundfile as sf
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
from pathlib import Path
import pandas as pd

ADAPTER_DIR = '/content/HDSD/whisper-adalora-hdsd/final_adapter'
MODEL_ID    = 'openai/whisper-small'

base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
finetuned = PeftModel.from_pretrained(base, ADAPTER_DIR)
finetuned.eval()

processor = WhisperProcessor.from_pretrained(ADAPTER_DIR)

# Pick a test sample
test_df = pd.read_csv('/content/HDSD/test.csv').head(3)
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
finetuned = finetuned.to(device)

print(f"Inference device: {device}\n")
for _, row in test_df.iterrows():
    audio, _ = sf.read(row['path'], dtype='float32', always_2d=False)
    feats = processor.feature_extractor(
        audio, sampling_rate=16000, return_tensors='pt'
    ).input_features.to(device)
    with torch.no_grad():
        ids = finetuned.generate(
            feats,
            forced_decoder_ids=processor.get_decoder_prompt_ids(
                language='hi', task='transcribe'),
        )
    pred = processor.tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f"  REF : {row['transcript']}")
    print(f"  HYP : {pred}")
    print()